In [1]:
# Simulated live flight schedule & gate database
MOCK_FLIGHTS = {
    "6E-204": {
        "route": "Delhi (DEL) to Mumbai (BOM)",
        "time": "14:30",
        "status": "ON TIME",
        "gate": "14",
        "terminal": "T2"
    },
    "6E-551": {
        "route": "Bengaluru (BLR) to Goa (GOI)",
        "time": "18:00",
        "status": "DELAYED (New departure: 18:45)",
        "gate": "06",
        "terminal": "T1"
    }
}

# Simulated PNR booking database
MOCK_BOOKINGS = {
    "IND123": {
        "name": "Rahul Sharma",
        "flight": "6E-204",
        "route": "DEL -> BOM",
        "seat": "12A",
        "status": "CONFIRMED"
    },
    "FLY789": {
        "name": "Priya Patel",
        "flight": "6E-551",
        "route": "BLR -> GOI",
        "seat": "07C",
        "status": "CHECKED-IN"
    }
}

# Simulated fare pricing database
MOCK_PRICES = {
    "DEL-BOM": "₹4,499 (Standard) / ₹6,299 (Flexi Plus)",
    "BOM-GOI": "₹3,150 (Standard) / ₹4,800 (Flexi Plus)",
    "BLR-DEL": "₹5,200 (Standard) / ₹7,100 (Flexi Plus)"
}

In [2]:
import random

def mock_airline_service(intent, user_message):
    msg_upper = user_message.upper()
    
    # Branch A: Flight Status & Schedule
    if "status" in intent or "schedule" in intent:
        for flight_no, details in MOCK_FLIGHTS.items():
            if flight_no in msg_upper or flight_no.replace("-", "") in msg_upper:
                return (
                    f"Dear Passenger,\n\n"
                    f"Thank you for contacting IndiGo regarding flight {flight_no}.\n\n"
                    f"Here are your latest flight details:\n"
                    f"• Flight Number: {flight_no}\n"
                    f"• Route: {details['route']}\n"
                    f"• Scheduled Departure: {details['time']}\n"
                    f"• Current Status: {details['status']}\n"
                    f"• Terminal & Gate: Terminal {details['terminal']}, Gate {details['gate']}\n\n"
                    f"Please arrive at the boarding gate at least 25 minutes prior to departure.\n\n"
                    f"Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
                )
        # Fallback if no specific flight number was mentioned
        return (
            "Dear Passenger,\n\n"
            "Thank you for contacting IndiGo flight status service.\n\n"
            "Our scheduled domestic flights are currently operating on schedule. "
            "To view the real-time status of your specific flight, please reply with your "
            "flight number (e.g., 6E-204) or visit www.goindigo.in/flight-status.\n\n"
            "Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
        )
    
    # Branch B: PNR & Reservation Details
    elif "reservation" in intent or "details" in intent:
        for pnr, b in MOCK_BOOKINGS.items():
            if pnr in msg_upper:
                return (
                    f"Dear {b['name']},\n\n"
                    f"Thank you for contacting IndiGo reservations.\n\n"
                    f"Here are your confirmed reservation details (PNR: {pnr}):\n"
                    f"• Passenger Name: {b['name']}\n"
                    f"• Flight: {b['flight']} ({b['route']})\n"
                    f"• Seat: {b['seat']}\n"
                    f"• Status: {b['status']}\n\n"
                    f"Web check-in opens 48 hours prior to departure.\n\n"
                    f"Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
                )
        return (
            "Dear Passenger,\n\n"
            "We could not locate an active booking with the reference provided. "
            "Please verify your 6-character PNR (e.g., IND123) and reply to this email.\n\n"
            "Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
        )
        
    # Branch C: Fare & Price Quotes
    elif "prices" in intent:
        return (
            f"Dear Passenger,\n\n"
            f"Thank you for inquiring about IndiGo flight fares.\n\n"
            f"Current starting fares on popular routes:\n"
            f"• Delhi to Mumbai: {MOCK_PRICES['DEL-BOM']}\n"
            f"• Mumbai to Goa: {MOCK_PRICES['BOM-GOI']}\n"
            f"• Bengaluru to Delhi: {MOCK_PRICES['BLR-DEL']}\n\n"
            f"Fares include standard 7 kg cabin baggage and 15 kg checked baggage.\n\n"
            f"Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
        )
        
    # Branch D: Booking Simulator
    else:
        new_pnr = "6E" + str(random.randint(1000, 9999))
        return (
            f"Dear Passenger,\n\n"
            f"Thank you for initiating your flight booking with IndiGo.\n\n"
            f"Your provisional reservation has been generated:\n"
            f"• Provisional PNR: {new_pnr}\n"
            f"• Booking Status: Awaiting Payment\n\n"
            f"Please complete your reservation within 2 hours at www.goindigo.in/pay.\n\n"
            f"Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
        )

In [3]:
def generate_email_subject(intent, resolved=True):
    # Convert 'check_baggage_allowance' -> 'Baggage Allowance'
    clean_topic = intent.replace("check_", "").replace("get_", "").replace("_", " ").title()
    
    if resolved:
        # Case A: Email sent to Customer
        return f"IndiGo Customer Support: Your Inquiry Regarding {clean_topic}"
    else:
        # Case B: Internal Ticket sent to rautom41105@gmail.com
        return f"[ESCALATION REQUIRED] Support Ticket: {clean_topic}"

In [4]:
import requests

N8N_PRODUCTION_URL = "https://intern.app.n8n.cloud/webhook/fa48a815-ee95-4229-a17e-5c38f6d28530"

def trigger_n8n_webhook(customer_email, subject, answer, resolved):
    payload = {
        "customer_email": customer_email,
        "subject": subject,
        "answer": answer,
        "resolved": resolved    # True (Customer Email) or False (rautom41105@gmail.com)
    }
    
    try:
        response = requests.post(N8N_PRODUCTION_URL, json=payload, timeout=10)
        print(f"[n8n Webhook Status]: HTTP {response.status_code}")
        return response.status_code == 200
    except Exception as e:
        print(f"[n8n Webhook Error]: {e}")
        return False